In [ ]:
from ultralytics import YOLO
import os
from IPython.display import Image as IPImage, display
from torchvision.ops import box_iou
from PIL import Image
import torch
import time
from pathlib import Path
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.optim as optim
from torchvision.ops import box_iou
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import shutil
from shutil import copyfile
import matplotlib.pyplot as plt
import random
from matplotlib import animation, rc
import yaml

## YOLO

In [ ]:
# Load YOLOv8s pretrained model
model = YOLO("yolov8s.pt")

# Fine-tune on Nepali license plate dataset with augmentations
model.train(
    data=r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    name="nepali_plate_yolov8s_augmented",
    project=r"PyTorch\runs\finetuned",
    augment=True  
)


#  Evaluate the trained model
metrics = model.val(
    data=r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\data.yaml",
    save=True
)

# Predict on images from the same dataset
results = model.predict(
    source=r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\images",
    save=True,
    conf=0.5
)

# Visualize one prediction
save_dir = results[0].save_dir
result_imgs = os.listdir(save_dir)
if result_imgs:
    first_img = os.path.join(save_dir, result_imgs[0])
    display(IPImage(filename=first_img))
else:
    print("No predicted images found.")


In [ ]:
# Load fine-tuned YOLOv8s model 
model = YOLO(r"\runs\finetuned\nepali_plate_yolov8s_augmented\weights\best.pt")


# Paths
img_dir = Path(r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\images")
label_dir = Path(r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\labels")

# Metrics
correct, iou_list, times = 0, [], []
num_images = 0
iou_threshold = 0.5  # IoU threshold for a prediction to be counted as correct

# Evaluation loop
for img_path in img_dir.glob("*.jpg"):
    label_path = label_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        continue

    with open(label_path, 'r') as f:
        line = f.readline().strip()
        if not line:
            continue
        cls, x, y, w, h = map(float, line.split())

    img = Image.open(img_path)
    iw, ih = img.size
    x1 = (x - w / 2) * iw
    y1 = (y - h / 2) * ih
    x2 = (x + w / 2) * iw
    y2 = (y + h / 2) * ih
    gt_box = torch.tensor([[x1, y1, x2, y2]])

    # Run inference 
    start = time.time()
    results = model.predict(str(img_path), conf=0.25, verbose=False)
    times.append(time.time() - start)

    if len(results[0].boxes) == 0:
        continue

    pred_box = results[0].boxes.xyxy[0].unsqueeze(0).cpu()
    pred_class = int(results[0].boxes.cls[0].item())

    iou = box_iou(pred_box, gt_box).item()
    iou_list.append(iou)
    num_images += 1

    if pred_class == int(cls) and iou >= iou_threshold:
        correct += 1

# Report 
print("✅ YOLOv8 Evaluation Results")
print(f"Images Evaluated: {num_images}")
print(f"Accuracy (IoU ≥ {iou_threshold}): {correct / num_images:.4f}")
print(f"Avg IoU: {sum(iou_list) / len(iou_list):.4f}")
print(f"Avg Inference Time: {sum(times) / len(times):.4f} sec")


## FasterRCNN

In [ ]:
class NepaliPlateDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = Path(img_dir)
        self.label_dir = Path(label_dir)
        self.transforms = transforms
        self.images = sorted(self.img_dir.glob("*.jpg"))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label_path = self.label_dir / (img_path.stem + ".txt")
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        boxes, labels = [], []
        with open(label_path) as f:
            for line in f:
                cls, x, y, w, h = map(float, line.strip().split())
                x_min = max((x - w / 2) * W, 0)
                y_min = max((y - h / 2) * H, 0)
                x_max = min((x + w / 2) * W, W)
                y_max = min((y + h / 2) * H, H)

                # Only append valid boxes with positive width and height
                if (x_max - x_min) > 1 and (y_max - y_min) > 1:
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(cls) + 1)  # class 0 = background in Faster R-CNN

        #  If no valid boxes found, skip and pick the next item (loop if at end)
        if not boxes:
            return self.__getitem__((idx + 1) % len(self))

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64)
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target



from torchvision.transforms import functional as F
from torch.utils.data import DataLoader

def get_transform():
    return transforms.Compose([
        transforms.Resize((640, 640)),
        transforms.ToTensor()
    ])

img_dir = r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\images"
label_dir = r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\labels"

dataset = NepaliPlateDataset(img_dir, label_dir, transforms=get_transform())
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))


In [ ]:
#  Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = fasterrcnn_resnet50_fpn(weights="DEFAULT")  # Pretrained COCO weights
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
model.to(device)

# Freeze backbone initially 
def freeze_backbone(model):
    for param in model.backbone.parameters():
        param.requires_grad = False
    return model

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True
    return model

model = freeze_backbone(model)  # Freeze backbone for warm-up

# === Optimizer & LR Scheduler ===
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# === Training Loop ===
num_epochs = 20
for epoch in range(num_epochs):
    if epoch == 5:  # 🔓 Unfreeze after warm-up
        model = unfreeze_all(model)
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
        print("🔓 Unfroze backbone for full fine-tuning.")

    model.train()
    total_loss = 0.0

    for images, targets in dataloader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        total_loss += losses.item()

    lr_scheduler.step()
    print(f"[Epoch {epoch+1}] Loss: {total_loss:.4f}")
    for k, v in loss_dict.items():
        print(f"   - {k}: {v.item():.4f}")

torch.save(model.state_dict(), r"runs\finetuned\fasterrcnn_nepali_plate.pth")


In [ ]:
# Load the trained model 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build model and load trained weights
model = fasterrcnn_resnet50_fpn(weights=None)  # no pretrained COCO
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
model.load_state_dict(torch.load(r"runs\finetuned\fasterrcnn_nepali_plate.pth"))  # path to saved weights
model.to(device).eval()

# --- Evaluation paths ---
img_dir = Path(r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\images")
label_dir = Path(r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\labels")

# --- Evaluation metrics ---
correct, iou_list, times = 0, [], []
num_images = 0
iou_threshold = 0.5
transform = transforms.Compose([transforms.Resize((640, 640)), transforms.ToTensor()])

for img_path in img_dir.glob("*.jpg"):
    label_path = label_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        continue

    with open(label_path, 'r') as f:
        line = f.readline().strip()
        if not line:
            continue
        cls, x, y, w, h = map(float, line.split())

    img = Image.open(img_path).convert("RGB")
    iw, ih = img.size
    x1 = (x - w / 2) * iw
    y1 = (y - h / 2) * ih
    x2 = (x + w / 2) * iw
    y2 = (y + h / 2) * ih
    gt_box = torch.tensor([[x1, y1, x2, y2]]).to(device)
    gt_class = int(cls) + 1  # because background is 0

    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        start = time.time()
        outputs = model(img_tensor)
        times.append(time.time() - start)

    if len(outputs[0]['boxes']) == 0:
        continue

    pred_box = outputs[0]['boxes'][0].unsqueeze(0)
    pred_class = outputs[0]['labels'][0].item()

    iou = box_iou(pred_box, gt_box).item()
    iou_list.append(iou)
    num_images += 1

    if pred_class == gt_class and iou >= iou_threshold:
        correct += 1

# Print results
print("✅ Faster R-CNN Evaluation Results")
print(f"Images Evaluated: {num_images}")
print(f"Accuracy (IoU ≥ {iou_threshold}): {correct / num_images:.4f}")
print(f"Avg IoU: {sum(iou_list) / len(iou_list):.4f}")
print(f"Avg Inference Time: {sum(times) / len(times):.4f} sec")


## FasterRCNN from Scratch

In [ ]:
num_classes = 2  # 1 class (number plate) + background
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load base model without pretrained weights
model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None)

# Replace classifier head
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.to(device)


In [ ]:
def get_transform(train=True):
    base = [
        transforms.Resize((640, 640)),
        transforms.ToTensor()
    ]
    if train:
        aug = [
            transforms.ColorJitter(brightness=0.3, contrast=0.3),
            transforms.RandomHorizontalFlip(0.5)
        ]
        return transforms.Compose(aug + base)
    else:
        return transforms.Compose(base)

In [ ]:
train_dataset = NepaliPlateDataset(img_dir, label_dir, transforms=get_transform(train=True))
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

optimizer = optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    lr_scheduler.step()
    print(f"[Epoch {epoch+1}] Loss: {total_loss:.4f}")
    for k, v in loss_dict.items():
        print(f"   - {k}: {v.item():.4f}")

In [ ]:
# Load model and weights
model.eval()
model.load_state_dict(torch.load(r"runs\finetuned\fasterrcnn_nepali_plate_scratch.pth"))
model.to(device)

# Paths
img_dir = Path(r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\images")
label_dir = Path(r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\labels")

# Metrics
correct, iou_list, times = 0, [], []
num_images = 0
iou_threshold = 0.5

for img_path in img_dir.glob("*.jpg"):
    label_path = label_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        continue

    with open(label_path, 'r') as f:
        line = f.readline().strip()
        if not line:
            continue
        cls, x, y, w, h = map(float, line.split())

    img = Image.open(img_path).convert("RGB")
    iw, ih = img.size
    x1 = (x - w / 2) * iw
    y1 = (y - h / 2) * ih
    x2 = (x + w / 2) * iw
    y2 = (y + h / 2) * ih
    gt_box = torch.tensor([[x1, y1, x2, y2]]).to(device)

    # Inference
    start = time.time()
    with torch.no_grad():
        outputs = model([transforms.ToTensor()(img).to(device)])
    times.append(time.time() - start)

    if len(outputs[0]['boxes']) == 0:
        continue

    pred_box = outputs[0]['boxes'][0].unsqueeze(0)
    pred_class = int(outputs[0]['labels'][0].item())

    iou = box_iou(pred_box, gt_box).item()
    iou_list.append(iou)
    num_images += 1

    if pred_class == int(cls + 1) and iou >= iou_threshold:
        correct += 1

# Report
print("✅ Faster R-CNN (Scratch) Evaluation Results")
print(f"Images Evaluated: {num_images}")
print(f"Accuracy (IoU ≥ {iou_threshold}): {correct / num_images:.4f}")
print(f"Avg IoU: {sum(iou_list) / len(iou_list):.4f}")
print(f"Avg Inference Time: {sum(times) / len(times):.4f} sec")


## YOLO implementation(kagglehub)

In [ ]:
img_path = Path(r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\images")
label_path = Path(r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\open_source_nepali_plates\labels")


In [ ]:
ipaths0=[]
types=[]
for dirname, _, filenames in os.walk(img_path):
    for filename in filenames:
        ipaths0+=[os.path.join(dirname, filename)]
        types+=[filename.split('.')[-1]]
tpaths0=[]
for dirname, _, filenames in os.walk(label_path):
    for filename in filenames:
        tpaths0+=[os.path.join(dirname, filename)]    
ipaths0=sorted(ipaths0)
tpaths0=sorted(tpaths0)
paths=[]
for ip,tp in zip(ipaths0,tpaths0):
    paths+=[(ip,tp)]
random.shuffle(paths)
ipaths=[]
tpaths=[]
for p in paths[0:200]:
    ipaths+=[p[0]]
    tpaths+=[p[1]]
print(len(ipaths))

In [ ]:
print(set(types))
for typei in list(set(types)):
    print(typei,types.count(typei))

In [ ]:
boxdata=[]
boxfile=[]
for i in range(len(tpaths)):
    tfile=tpaths[i]
    ifile=ipaths[i]
    boxdata+=[np.loadtxt(tfile)]     
    boxfile+=[ifile.split('/')[-1]]
print(boxdata[0])

In [ ]:
BOX=pd.DataFrame()

for i in range(len(boxdata)):
    if type(boxdata[i][0])==np.float64:
        add=pd.DataFrame([boxdata[i]])
        add[5]=boxfile[i]
        BOX=pd.concat([BOX,add])
    else:
        add=pd.DataFrame(boxdata[i])
        add[5]=boxfile[i]
        BOX=pd.concat([BOX,add])       

BOX2=BOX.reset_index(drop=True)
BOX2.iloc[:,0]=BOX2.iloc[:,0].astype(int)
display(BOX2)

In [ ]:
display(BOX2.iloc[:,0].value_counts())
BOX2[5].value_counts()

In [ ]:
def draw_box(n0):
    
    ipath=ipaths[n0]
    image=cv2.imread(ipath)
    H,W=image.shape[0],image.shape[1]
    file=ipath.split('/')[-1] 
    
    if BOX2[BOX2[5]==file] is not None:
        box=BOX2[BOX2[5]==file]
        box=box.reset_index(drop=True)
        #display(box)
        
        for i in range(len(box)):
            label=int(box.loc[i,0])
            x=box.loc[i,1]
            y=box.loc[i,2]
            w=box.loc[i,3] 
            h=box.loc[i,4]
            x1=((x-w/2)*W).astype(int)
            y1=((y-h/2)*H).astype(int)
            x2=((x+w/2)*W).astype(int)
            y2=((y+h/2)*H).astype(int)
            
            cv2.rectangle(image,(x1,y1),(x2,y2),(0,255,0),1) #green
    return image


In [ ]:
images1=[]
for i in range(len(ipaths)):
    images1+=[draw_box(i)]

rc('animation', html='jshtml')

def create_animation(ims):    
    fig=plt.figure(figsize=(10,6))
    #plt.axis('off')
    im=plt.imshow(cv2.cvtColor(ims[0],cv2.COLOR_BGR2RGB))
    plt.close()    
    def animate_func(i):
        im.set_array(cv2.cvtColor(ims[i],cv2.COLOR_BGR2RGB))
        return [im]
    return animation.FuncAnimation(fig, animate_func, frames=len(ims), interval=1000//2)

create_animation(images1)

In [ ]:
# Set your original dataset paths 
img_dir = Path(r"kagglehub/datasets/inspiring-lab/nepali-vehicles-number-plate-dataset/versions/1/open_source_nepali_plates/images")
label_dir = Path(r"kagglehub/datasets/inspiring-lab/nepali-vehicles-number-plate-dataset/versions/1/open_source_nepali_plates/labels")

# Set destination root directory 
dest_root = Path(r"kagglehub/datasets/inspiring-lab/nepali-vehicles-number-plate-dataset/versions/1/split")

# Create train/valid/test image & label directories 
for split in ['train', 'valid', 'test']:
    os.makedirs(dest_root / split / "images", exist_ok=True)
    os.makedirs(dest_root / split / "labels", exist_ok=True)

# Get all .jpg image-label pairs 
ipaths = sorted(img_dir.glob("*.jpg"))
tpaths = [label_dir / (img_path.stem + ".txt") for img_path in ipaths]

# Copy files to respective folders 
for i in range(len(ipaths)):
    split = "train" if i % 3 == 0 else "valid" if i % 3 == 1 else "test"

    img_dest = dest_root / split / "images" / ipaths[i].name
    lbl_dest = dest_root / split / "labels" / tpaths[i].name

    copyfile(ipaths[i], img_dest)
    if tpaths[i].exists():
        copyfile(tpaths[i], lbl_dest)

print("✅ Dataset split into train/valid/test folders successfully.")


In [ ]:
#  Corrected paths for your dataset 
face_yaml = dict(
    train = r"kagglehub/datasets/inspiring-lab/nepali-vehicles-number-plate-dataset/versions/1/split/train/images",
    val   = r"kagglehub/datasets/inspiring-lab/nepali-vehicles-number-plate-dataset/versions/1/split/valid/images",
    test  = r"kagglehub/datasets/inspiring-lab/nepali-vehicles-number-plate-dataset/versions/1/split/test/images",
    nc    = 1,
    names = ['character']
)

# Save YAML to the same root directory
yaml_path = r"kagglehub/datasets/inspiring-lab/nepali-vehicles-number-plate-dataset/versions/1/split/face.yaml"

# Create parent directory if needed
os.makedirs(os.path.dirname(yaml_path), exist_ok=True)

# Write to face.yaml
with open(yaml_path, 'w') as f:
    yaml.dump(face_yaml, f)

print(f"✅ YOLO config saved at: {yaml_path}")


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=r"kagglehub/datasets/inspiring-lab/nepali-vehicles-number-plate-dataset/versions/1/split/face.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    project=r"runs",
    name="nepali_plate_split_yolov8n"
)


In [ ]:
tpaths2=[]
for dirname, _, filenames in os.walk('runs/nepali_plate_split_yolov8n'):
    for filename in filenames:
        if filename[-4:]=='.png' or filename[-4:]=='.jpg':
            tpaths2+=[(os.path.join(dirname, filename))]
tpaths2=sorted(tpaths2)
print(tpaths2[0])

for path in tpaths2:
    image = Image.open(path)
    image=np.array(image)
    plt.figure(figsize=(20,10))
    plt.imshow(image)
    plt.show()


In [ ]:
# Path to your fine-tuned model weights
model_path = r"runs\nepali_plate_split_yolov8n\weights\best.pt"

# Path to your test image directory
test_dir = Path(r"\kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\split\test\images")

#  Load model
model = YOLO(model_path)

# Run prediction
results = model.predict(
    source=str(test_dir),
    conf=0.05,
    save=True,
    project=r"runs\predictions",
    name="nepali_plate_split_yolov8n"
)

# ✅ Output prediction path
print(f"✅ Predictions saved to: {results[0].save_dir}")


In [ ]:
# Rebuild PBOX from results 
PBOX = pd.DataFrame(columns=range(6))

for i, r in enumerate(results):
    arr = pd.DataFrame(r.boxes.data.cpu()).astype(float)
    file = Path(r.path).name  # get the file name from YOLO result
    arr = arr.assign(file=file)
    arr = arr.assign(i=i)
    PBOX = pd.concat([PBOX, arr], axis=0)

PBOX.columns = ['x1', 'y1', 'x2', 'y2', 'confidence', 'class', 'file', 'i']
PBOX['class'] = PBOX['class'].astype(int)

# ✅ Display result
display(PBOX.head())


In [ ]:
# Set your prediction output and test images path
pred_dir = r"runs\predictions\nepali_plate_split_yolov8n"
test_img_dir = r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\split\test\images"

# Get all prediction image paths
ppaths = []
for root, _, files in os.walk(test_img_dir):
    for file in files:
        if file.lower().endswith(".jpg"):
            ppaths.append(os.path.join(root, file))
ppaths = sorted(ppaths)

# Load model predictions from previous results
from ultralytics import YOLO
model = YOLO(r"runs\nepali_plate_split_yolov8n\weights\best.pt")
results = model.predict(source=test_img_dir, conf=0.05, verbose=False)

# Build dataframe for predictions
PBOX = pd.DataFrame(columns=range(6))
for i in range(len(results)):
    arr = pd.DataFrame(results[i].boxes.data.cpu()).astype(float)
    file = os.path.basename(ppaths[i])
    arr = arr.assign(file=file)
    arr = arr.assign(i=i)
    PBOX = pd.concat([PBOX, arr], axis=0)
PBOX.columns = ['x1', 'y1', 'x2', 'y2', 'confidence', 'class', 'file', 'i']
PBOX['class'] = PBOX['class'].astype(int)
PBOX = PBOX.reset_index(drop=True)


In [ ]:
def draw_box2(n0):
    ipath = ppaths[n0]
    image = cv2.imread(ipath)
    H, W = image.shape[:2]
    file = os.path.basename(ipath)

    if file in PBOX['file'].values:
        box = PBOX[PBOX['file'] == file].reset_index(drop=True)
        for i in range(len(box)):
            x1 = int(box.loc[i, 'x1'])
            y1 = int(box.loc[i, 'y1'])
            x2 = int(box.loc[i, 'x2'])
            y2 = int(box.loc[i, 'y2'])
            label = int(box.loc[i, 'class'])
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 1)
            cv2.putText(image, str(label), (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Collect images
images2 = [draw_box2(i) for i in tqdm(range(len(ppaths)))]


In [ ]:
def create_animation(images, save_path="predictions.gif"):
    fig = plt.figure()
    ims = [[plt.imshow(img, animated=True)] for img in images]

    ani = animation.ArtistAnimation(fig, ims, interval=300, blit=True, repeat_delay=1000)
    ani.save(save_path)
    print(f"✅ Animation saved at: {save_path}")

create_animation(images2)


In [ ]:
label_dir = Path(r"kagglehub\datasets\inspiring-lab\nepali-vehicles-number-plate-dataset\versions\1\split\test\labels")

correct, iou_list, times = 0, [], []
iou_threshold = 0.5
num_images = 0

for i, img_path in enumerate(ppaths):
    label_path = label_dir / (Path(img_path).stem + ".txt")
    if not label_path.exists():
        continue

    with open(label_path) as f:
        line = f.readline().strip()
        if not line:
            continue
        cls, x, y, w, h = map(float, line.split())

    img = Image.open(img_path)
    iw, ih = img.size
    x1 = (x - w / 2) * iw
    y1 = (y - h / 2) * ih
    x2 = (x + w / 2) * iw
    y2 = (y + h / 2) * ih
    gt_box = torch.tensor([[x1, y1, x2, y2]])

    start = time.time()
    result = model.predict(img_path, conf=0.25, verbose=False)[0]
    times.append(time.time() - start)

    if len(result.boxes) == 0:
        continue

    pred_box = result.boxes.xyxy[0].unsqueeze(0).cpu()
    pred_class = int(result.boxes.cls[0].item())

    iou = box_iou(pred_box, gt_box).item()
    iou_list.append(iou)
    num_images += 1

    if pred_class == int(cls) and iou >= iou_threshold:
        correct += 1

# Print metrics
print("✅ YOLOv8 Validation Metrics")
print(f"Images Evaluated: {num_images}")
print(f"Accuracy (IoU ≥ {iou_threshold}): {correct / num_images:.4f}")
print(f"Avg IoU: {sum(iou_list)/len(iou_list):.4f}")
print(f"Avg Inference Time: {sum(times)/len(times):.4f} sec")
